# Object tracking using openCV

# 1.To get a Video Clip 

In [ ]:
import cv2
import time

cap = cv2.VideoCapture(0)
start_time = time.time()

fourcc = cv2.VideoWriter_fourcc(*'XVID')
out = cv2.VideoWriter('output.avi', fourcc, 20.0, (640, 480))

if not cap.isOpened():
    print("Error: Unable to access the camera.")
    exit()

print("Recording video. Press 'q' to quit.")
while True:
    ret, frame = cap.read()
    if not ret:
        print("Failed to capture frame. Exiting...")
        break

    out.write(frame)
    cv2.imshow('Video Capture', frame)

    if cv2.waitKey(1) & 0xFF == ord('q') or time.time() - start_time > 30:
        break

cap.release()
out.release()
cv2.destroyAllWindows()
print("Video recording complete and saved as 'output.avi'.")

# 2.Object tracking using CV2

In [ ]:
import cv2
from tracker import *

tracker = EuclideanDistTracker()
cap = cv2.VideoCapture("highway.mp4")
object_detector = cv2.createBackgroundSubtractorMOG2(history=150, varThreshold=20)

while True:
    ret, frame = cap.read()
    height, width, _ = frame.shape

    roi = frame[150: 550,100: 600]

    # 1. Object Detection
    mask = object_detector.apply(roi)
    _, mask = cv2.threshold(mask, 254, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(mask, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
    detections = []
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area > 100:
            x, y, w, h = cv2.boundingRect(cnt)
            detections.append([x, y, w, h])
    print("detections" , detections)
    
    # 2. Object Tracking
    boxes_ids = tracker.update(detections)
    for box_id in boxes_ids:
        x, y, w, h, id = box_id
        cv2.putText(roi, str(id), (x, y - 15), cv2.FONT_HERSHEY_PLAIN, 2, (255, 0, 0), 2)
        cv2.rectangle(roi, (x, y), (x + w, y + h), (0, 255, 0), 3)

    cv2.imshow("roi", roi)
    cv2.imshow("Frame", frame)
    # cv2.imshow("Mask", mask)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        cv2.destroyAllWindows()
        break


# 3.Face detection and moving face countouring along with stationary faces

In [ ]:
import cv2
import mediapipe as mp

mp_face_detection = mp.solutions.face_detection
mp_drawing = mp.solutions.drawing_utils

cap = cv2.VideoCapture(0)

with mp_face_detection.FaceDetection(min_detection_confidence=0.5) as face_detection:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Convert the frame to RGB
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = face_detection.process(frame_rgb)

        if results.detections:
            for detection in results.detections:
                bboxC = detection.location_data.relative_bounding_box
                ih, iw, _ = frame.shape
                x, y, w, h = int(bboxC.xmin * iw), int(bboxC.ymin * ih), \
                             int(bboxC.width * iw), int(bboxC.height * ih)
                cv2.rectangle(frame, (x, y), (x + w, y + h), (255, 0, 0), 3)

        cv2.imshow('Face Detection', frame)
        if cv2.waitKey(50) & 0xFF == ord('q'):
            cap.release()
            cv2.destroyAllWindows()
            break


# 4.Face detection and movement face contouring

In [ ]:
import cv2
import mediapipe as mp

mp_face_detection = mp.solutions.face_detection
mp_drawing = mp.solutions.drawing_utils
face_detector = mp_face_detection.FaceDetection(min_detection_confidence=0.5)

object_detector = cv2.createBackgroundSubtractorMOG2(history=100, varThreshold=40)

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.resize(frame, (640, 480))
    height, width, _ = frame.shape

    mask = object_detector.apply(frame)
    _, mask = cv2.threshold(mask, 254, 255, cv2.THRESH_BINARY)

    contours, _ = cv2.findContours(mask, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

    motion_boxes = []
    for cnt in contours:
        if cv2.contourArea(cnt) > 1000:  # Filter small motions
            x, y, w, h = cv2.boundingRect(cnt)
            motion_boxes.append((x, y, w, h))

    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = face_detector.process(frame_rgb)

    if results.detections:
        for detection in results.detections:
            # Extract face bounding box
            bboxC = detection.location_data.relative_bounding_box
            x, y, w, h = int(bboxC.xmin * width), int(bboxC.ymin * height), \
                         int(bboxC.width * width), int(bboxC.height * height)

            for mx, my, mw, mh in motion_boxes:
                if (x < mx + mw and x + w > mx and
                        y < my + mh and y + h > my):

                    cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 3)
                    cv2.putText(frame, "Moving Face", (x, y - 10),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)

    cv2.imshow("Frame", frame)
    cv2.imshow("Motion Mask", mask)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        cap.release()
        cv2.destroyAllWindows()
        break


# 5.Moving face contouring and tracking over a threshold to get croppped frames

In [ ]:
import cv2
import mediapipe as mp

mp_face_detection = mp.solutions.face_detection
face_detector = mp_face_detection.FaceDetection(min_detection_confidence=0.7)

object_detector = cv2.createBackgroundSubtractorMOG2(history=100, varThreshold=40)

cap = cv2.VideoCapture(0)

trackers = cv2.legacy.MultiTracker()
tracker_id_counter = 1
tracked_faces = {}

line_x = 480

def calculate_iou(box1, box2):
    x1, y1, w1, h1 = box1
    x2, y2, w2, h2 = box2

    # Calculate intersection coordinates
    inter_x1 = max(x1, x2)
    inter_y1 = max(y1, y2)
    inter_x2 = min(x1 + w1, x2 + w2)
    inter_y2 = min(y1 + h1, y2 + h2)

    # Calculate intersection area
    inter_area = max(0, inter_x2 - inter_x1) * max(0, inter_y2 - inter_y1)

    # Calculate union area
    box1_area = w1 * h1
    box2_area = w2 * h2
    union_area = box1_area + box2_area - inter_area

    if union_area == 0:
        return 0
    return inter_area / union_area

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.resize(frame, (640, 480))
    height, width, _ = frame.shape

    mask = object_detector.apply(frame)
    _, mask = cv2.threshold(mask, 254, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(mask, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

    # Detect Motion Bounding Boxes
    motion_boxes = []
    for cnt in contours:
        if cv2.contourArea(cnt) > 1000:
            x, y, w, h = cv2.boundingRect(cnt)
            motion_boxes.append((x, y, w, h))

    # Detect Faces in the Current Frame
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = face_detector.process(frame_rgb)

    if results.detections:
        for detection in results.detections:
            # Get Face Bounding Box
            bboxC = detection.location_data.relative_bounding_box
            x, y, w, h = int(bboxC.xmin * width), int(bboxC.ymin * height), \
                         int(bboxC.width * width), int(bboxC.height * height)

            # Check Overlap with Motion Areas
            is_moving = any(
                mx < x + w and x < mx + mw and my < y + h and y < my + mh
                for mx, my, mw, mh in motion_boxes
            )
            if is_moving:
                # Check if face is already tracked
                already_tracked = False
                for tracked_id, data in tracked_faces.items():
                    tracked_box = data['box']
                    if calculate_iou((x, y, w, h), tracked_box) > 0.5:
                        already_tracked = True
                        break

                if not already_tracked:
                    tracker = cv2.legacy.TrackerCSRT_create()
                    trackers.add(tracker, frame, (x, y, w, h))
                    tracked_faces[tracker_id_counter] = {'crossed': False, 'box': (x, y, w, h)}
                    tracker_id_counter += 1

    ok, boxes = trackers.update(frame)
    for tracker_id, box in zip(tracked_faces.keys(), boxes):
        x, y, w, h = [int(v) for v in box]
        tracked_faces[tracker_id]['box'] = (x, y, w, h)

        # Check if Face Crossed the Line
        if x + w > line_x and not tracked_faces[tracker_id]['crossed']:
            print(f"Face {tracker_id} crossed the line!")
            tracked_faces[tracker_id]['crossed'] = True
            
            cropped_face = frame[y:y+h, x:x+w]
            file_name = f"user_{tracker_id}.jpg"
            cv2.imwrite(file_name, cropped_face)
            print(f"Saved cropped face as {file_name}")

        # Draw Bounding Box
        color = (0, 255, 0) if tracked_faces[tracker_id]['crossed'] else (0, 0, 255)
        cv2.rectangle(frame, (x, y), (x + w, y + h), color, 2)
        cv2.putText(frame, f"ID: {tracker_id} crossed: {tracked_faces[tracker_id]['crossed']}", (x, y - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

    cv2.line(frame, (line_x, 0), (line_x, height), (255, 255, 0), 2)

    cv2.imshow("Motion Mask", mask)
    cv2.imshow("Frame", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        cap.release()
        cv2.destroyAllWindows()
        break


# 6.Movement tracking and Face Detection either until a face is found or for duration of 5 seconds

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import cv2
import mediapipe as mp
import time
import os
from datetime import datetime

mp_face_detection = mp.solutions.face_detection
face_detector = mp.solutions.face_detection.FaceDetection(min_detection_confidence=0.7)

cap = cv2.VideoCapture(0)

detection_time = None
SAVE_DELAY = 2
no_face_start_time = time.time()
MAX_NO_FACE_TIME = 5

output_folder = "cropped_users"
os.makedirs(output_folder, exist_ok=True)
saved = False

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Convert frame to RGB for face detection
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = face_detector.process(frame_rgb)

    if results.detections:
        no_face_start_time = time.time()

        if detection_time is None:
            detection_time = time.time()

        elapsed_time = time.time() - detection_time

        if elapsed_time >= SAVE_DELAY:
            # Get the first detected face bounding box
            detection = results.detections[0]
            bboxC = detection.location_data.relative_bounding_box
            x, y, w, h = int(bboxC.xmin * frame.shape[1]), int(bboxC.ymin * frame.shape[0]), \
                         int(bboxC.width * frame.shape[1]), int(bboxC.height * frame.shape[0])

            # Crop the face
            x, y, w, h = max(0, x), max(0, y), max(0, w), max(0, h)
            x_end = min(frame.shape[1], x + w)
            y_end = min(frame.shape[0], y + h)

            cropped_face = frame[y:y_end, x:x_end]

            if cropped_face.size > 0:
                timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
                file_name = os.path.join(output_folder, f"user_{timestamp}.jpg")                
                cv2.imwrite(file_name, cropped_face)
                print(f"Saved cropped face as {file_name}")
                saved = True
                break
            else:
                print("Invalid cropped face. Skipping save.")
    else:
        detection_time = None

    # Check if no face has been detected for MAX_NO_FACE_TIME seconds
    elapsed_no_face_time = time.time() - no_face_start_time
    if elapsed_no_face_time >= MAX_NO_FACE_TIME:
        print("No face detected for 5 seconds. Closing the video capture.")
        break

    # Display the frame
    cv2.imshow("Frame", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
print("Resources released. Program exited.")
